# Feature selection — scFSNN (Feature Selection Neural Network với kiểm soát FDR)

Embedded method theo Peng et al., *BMC Genomics* 25:264 (2024). Mạng 2 lớp ẩn (256, 128) + BatchNorm + Dropout 0.5, Adam lr = 1e-3, batch 32. Thêm q = ⌊p/2⌋ **surrogate gene** (lấy mẫu ngẫu nhiên từ ma trận dữ liệu) để ước lượng FDR; importance S_j = trung bình |∂L/∂x_ij| trên tập holdout; p̂0 = min(2·#{S_j < median(S_surrogate)}, p); mỗi vòng xoá ⌈ε(1 − η*/η̂)·r0⌉ gene có S nhỏ nhất (ε = η* = 0.1), train tiếp 30 epoch (như code gốc github.com/linbingqing/scFSNN). Tăng cường dữ liệu convex pseudo-data (d = 0.2); khi train cộng nhiễu Gaussian vào đầu vào (η = 1, mặc định trong code gốc).

**Điều chỉnh để chọn đúng `N_FEATURES` gene:** bài báo tự dừng khi FDR ≤ η* (số gene thay đổi). Ở đây: (a) không cho số gene gốc xuống dưới `N_FEATURES`; (b) nếu đã đạt ngưỡng FDR mà còn nhiều hơn `N_FEATURES` gene thì tiếp tục loại gene gốc có S nhỏ nhất. Số gene tại thời điểm đạt ngưỡng FDR (lựa chọn "tự nhiên" của scFSNN) ghi ở cột `n_orig_at_fdr_cutoff`. Tập "validation" của bài báo được thay bằng 20% tách từ **train** của fold (không dùng val/test). **Tự dùng GPU nếu có** (Kaggle: Settings → Accelerator → GPU); chạy CPU sẽ rất chậm. Mặc định `PREPROCESS = "none"` (giữ log2 gốc, theo quy ước của đề tài). Lưu ý bài báo gốc có chuẩn hoá: log + z-score từng gene (công thức (1)). Có thể thử `PREPROCESS = "zscore"` để giống bài báo — chỉ áp bên trong bước chọn gene, dữ liệu gốc không đổi.

- **Đầu vào:** `DATA_DIR/fold_{1..5}/train.h5` (chỉ dùng tập **train**; val/test không đụng tới). Dữ liệu giữ nguyên log2 expression, không chuẩn hoá.
- **Đổi dataset:** sửa `DATA_DIR` ở cell cấu hình. Số gene chọn: `N_FEATURES` (mặc định 100).
- **Đầu ra** (`/kaggle/working/feature_selection/scfsnn/<dataset>/`):
  - `selected_genes/fold_k.csv`, `selected_genes_all_folds.csv`, `selected_genes.json` — gene được chọn theo fold (rank, score)
  - `metrics_per_fold.csv`, `metrics_summary.csv` — thời gian, CPU utilization, peak memory, energy, EDP, carbon footprint
- **Đo lường:** chỉ bao quanh lời gọi `select_features` (không tính thời gian đọc file). Energy/Carbon ước lượng bằng `codecarbon` (tracking_mode="process"); EDP = Energy(J) × Time(s).

In [ ]:
# tables: backend cho pd.read_hdf | codecarbon: do nang luong/CO2 | psutil: CPU & RAM
get_ipython().system("pip install -q codecarbon psutil tables")

In [ ]:
import os

# ====== CAU HINH (sua tay) ======
# Doi DATA_DIR sang bo khac (vd .../h5/tcga_luad) de chay dataset khac
DATA_DIR = "/kaggle/input/datasets/lfreedom2750/experiment-dataset/h5/gse68465"
N_FEATURES = 100          # so gene giu lai moi fold
N_FOLDS = 5
SEED = 42
METHOD = "scfsnn"
CLASS_WEIGHT = "balanced"      # None | "balanced": trong so lop trong cross-entropy
PREPROCESS = "none"            # "none" (log2 goc) | "l2_col" | "zscore" -- chi ben trong buoc chon gene
SCFSNN_HIDDEN = (256, 128)
SCFSNN_DROPOUT = 0.5
SCFSNN_LR = 1e-3
SCFSNN_BATCH = 32
SCFSNN_EPOCHS = 30            # epoch khoi tao va moi vong loai (code goc)
SCFSNN_FINAL_EPOCHS = 10      # train them tren tap gene cuoi (code goc)
SCFSNN_Q_RATIO = 0.5          # q = p/2 surrogate gene
SCFSNN_FDR = 0.1              # eta*
SCFSNN_ELIM_RATE = 0.1        # epsilon
SCFSNN_CPD_D = 0.2            # 0 = tat tang cuong du lieu
SCFSNN_INPUT_NOISE = 1.0      # eta: nhieu Gaussian them vao dau vao khi train (mac dinh code goc; bai bao khong nhac)
SCFSNN_HOLDOUT = 0.2         # ti le train dung lam holdout tinh importance/FDR
SCFSNN_MAX_ITER = 1000
METHOD_PARAMS = {"hidden": SCFSNN_HIDDEN, "dropout": SCFSNN_DROPOUT, "lr": SCFSNN_LR, "batch": SCFSNN_BATCH,
                 "epochs": SCFSNN_EPOCHS, "final_epochs": SCFSNN_FINAL_EPOCHS, "q_ratio": SCFSNN_Q_RATIO,
                 "fdr": SCFSNN_FDR, "elim_rate": SCFSNN_ELIM_RATE, "cpd_d": SCFSNN_CPD_D, "input_noise": SCFSNN_INPUT_NOISE,
                 "holdout": SCFSNN_HOLDOUT, "class_weight": CLASS_WEIGHT, "preprocess": PREPROCESS}
# None -> codecarbon tu dinh vi (can Internet); hoac dat ma ISO 3 ky tu, vd "USA", "VNM"
COUNTRY_ISO_CODE = None

DATASET_NAME = os.path.basename(os.path.normpath(DATA_DIR))
RESULT_DIR = os.path.join("/kaggle/working/feature_selection", METHOD, DATASET_NAME)
os.makedirs(os.path.join(RESULT_DIR, "selected_genes"), exist_ok=True)

assert os.path.isfile(os.path.join(DATA_DIR, "fold_1", "train.h5")), \
    f"Khong thay {DATA_DIR}/fold_1/train.h5 -- da Add Input dataset chua?"
print("Dataset :", DATASET_NAME, "->", DATA_DIR)
print("Method  :", METHOD, "| N_FEATURES =", N_FEATURES)
print("Ket qua :", RESULT_DIR)
print("So CPU  :", os.cpu_count())

## Đọc dữ liệu

In [ ]:
import json, time, threading, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}


def load_split(fold, split="train"):
    # File h5 (pandas, key="data"): index = sample_id, cot = ten gene + cot "label"
    df = pd.read_hdf(os.path.join(DATA_DIR, f"fold_{fold}", f"{split}.h5"), key="data")
    X = df.drop(columns="label")
    # Giu nguyen gia tri log2 expression, KHONG chuan hoa
    return X.values.astype(np.float64), df["label"].values.astype(int), X.columns.to_numpy(), df.index.to_numpy()


# Doc thu fold 1 de xem cau truc
X0, y0, genes0, ids0 = load_split(1)
print(f"fold_1/train: {X0.shape[0]} mau x {X0.shape[1]} gene | khoang gia tri [{X0.min():.2f}, {X0.max():.2f}]")
print("Vi du gene:", list(genes0[:5]), "| vi du sample:", list(ids0[:3]))
print("Phan bo lop:", pd.Series(y0).map(STAGE_NAMES).value_counts().to_dict())
del X0, y0, genes0, ids0

## Bộ đo tài nguyên

In [ ]:
import psutil

try:
    from codecarbon import EmissionsTracker, OfflineEmissionsTracker
    _CODECARBON = True
except ImportError:
    _CODECARBON = False


class ResourceTracker:
    """Do tren 1 khoi lenh: thoi gian (s), CPU utilization, peak memory (RSS, gom ca
    tien trinh con cua joblib), energy (kWh, codecarbon), carbon (kg CO2eq) va
    Energy-Delay Product EDP = Energy(J) * Time(s)."""

    def __init__(self, name, interval=0.05):
        self.name, self.interval = name, interval
        self.proc = psutil.Process()
        self._stop = threading.Event()

    def _procs(self):
        try:
            return [self.proc] + self.proc.children(recursive=True)
        except psutil.Error:
            return [self.proc]

    def _poll(self):
        # RSS tong va CPU time cua worker con (tru phan CPU worker da dung truoc khi bat dau do)
        rss = 0
        for p in self._procs():
            try:
                rss += p.memory_info().rss
                if p.pid != self.proc.pid:
                    t = p.cpu_times()
                    self._child_cpu[p.pid] = t.user + t.system - self._child_base.get(p.pid, 0.0)
            except psutil.Error:
                pass
        self._peak_rss = max(self._peak_rss, rss)
        return rss

    def _sampler(self):
        while not self._stop.is_set():
            self._poll()
            self._sys_cpu.append(psutil.cpu_percent(interval=None))
            self._stop.wait(self.interval)

    def __enter__(self):
        self._child_cpu, self._sys_cpu, self._peak_rss = {}, [], 0
        self._child_base = {}
        for p in self._procs()[1:]:
            try:
                t = p.cpu_times()
                self._child_base[p.pid] = t.user + t.system
            except psutil.Error:
                pass
        self.tracker = None
        if _CODECARBON:
            try:
                kw = dict(project_name=self.name, output_dir=RESULT_DIR, save_to_file=False,
                          log_level="error", measure_power_secs=1, tracking_mode="process")
                self.tracker = (OfflineEmissionsTracker(country_iso_code=COUNTRY_ISO_CODE, **kw)
                                if COUNTRY_ISO_CODE else EmissionsTracker(**kw))
                self.tracker.start()
            except Exception as e:
                print("[ResourceTracker] codecarbon loi:", e)
                self.tracker = None
        self.base_rss = self._poll()
        self._child_cpu = {}
        psutil.cpu_percent(interval=None)
        t = self.proc.cpu_times()
        self._cpu0 = t.user + t.system
        self._stop.clear()
        self._thread = threading.Thread(target=self._sampler, daemon=True)
        self._thread.start()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.time_sec = time.perf_counter() - self._t0
        t = self.proc.cpu_times()
        main_cpu = t.user + t.system - self._cpu0
        self._stop.set()
        self._thread.join(timeout=2)
        self._poll()
        cpu_sec = main_cpu + sum(self._child_cpu.values())

        self.energy_kwh, self.carbon_kg = float("nan"), float("nan")
        if self.tracker is not None:
            try:
                self.carbon_kg = float(self.tracker.stop() or 0.0)
                data = self.tracker.final_emissions_data
                self.energy_kwh = float(data.energy_consumed) if data is not None else float("nan")
            except Exception as e:
                print("[ResourceTracker] codecarbon stop loi:", e)

        n_cpu = os.cpu_count() or 1
        self.metrics = {
            "time_sec": self.time_sec,
            "cpu_time_sec": cpu_sec,
            # % tren tong nang luc may (100% = dung het n_cpu core)
            "cpu_util_percent": 100.0 * cpu_sec / (self.time_sec * n_cpu) if self.time_sec > 0 else float("nan"),
            "avg_cores_used": cpu_sec / self.time_sec if self.time_sec > 0 else float("nan"),
            "system_cpu_percent_mean": float(np.mean(self._sys_cpu)) if self._sys_cpu else float("nan"),
            "peak_memory_mb": self._peak_rss / 1024 ** 2,
            "peak_memory_increase_mb": (self._peak_rss - self.base_rss) / 1024 ** 2,
            "energy_kwh": self.energy_kwh,
            "energy_joule": self.energy_kwh * 3.6e6,
            "edp_joule_sec": self.energy_kwh * 3.6e6 * self.time_sec,
            "carbon_kg_co2eq": self.carbon_kg,
            "carbon_g_co2eq": self.carbon_kg * 1e3,
        }
        return False

## Thuật toán: scFSNN (Feature Selection Neural Network với kiểm soát FDR)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Tu dung GPU neu co (Kaggle: Settings -> Accelerator -> GPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None
print("Thiet bi:", DEVICE, "|", GPU_NAME or f"{os.cpu_count()} CPU", "| torch", torch.__version__)
if DEVICE.type == "cuda":
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SELECT_INFO = {}   # thong tin phu cua lan chon gan nhat, duoc ghi vao metrics_per_fold.csv


def _preprocess(X):
    # "none": giu log2 goc | "l2_col": chia moi gene cho chuan L2 (nhu code wsBNN) | "zscore": nhu scFSNN
    if PREPROCESS == "none":
        return X
    if PREPROCESS == "l2_col":
        nrm = np.linalg.norm(X, axis=0)
        nrm[nrm == 0] = 1.0
        return X / nrm
    if PREPROCESS == "zscore":
        sd = X.std(0)
        sd[sd == 0] = 1.0
        return (X - X.mean(0)) / sd
    raise ValueError(PREPROCESS)


def _class_weight(yc, n_class):
    if CLASS_WEIGHT != "balanced":
        return None
    cnt = np.bincount(yc, minlength=n_class)
    return torch.tensor(len(yc) / (n_class * cnt), dtype=torch.float32, device=DEVICE)


def _gpu_begin():
    SELECT_INFO.clear()
    SELECT_INFO.update(device=DEVICE.type, gpu_name=GPU_NAME)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def _gpu_end():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        SELECT_INFO["gpu_peak_memory_mb"] = torch.cuda.max_memory_allocated() / 1024 ** 2

from sklearn.model_selection import train_test_split


class FSNet(nn.Module):
    """Kien truc scFSNN: 2 lop an (256, 128) + BatchNorm + ReLU + Dropout(0.5), dau ra softmax."""

    def __init__(self, d, n_class):
        super().__init__()
        h1, h2 = SCFSNN_HIDDEN
        self.inp = nn.Linear(d, h1)
        self.body = nn.Sequential(nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(SCFSNN_DROPOUT),
                                  nn.Linear(h1, h2), nn.BatchNorm1d(h2), nn.ReLU(), nn.Dropout(SCFSNN_DROPOUT),
                                  nn.Linear(h2, n_class))

    def forward(self, x):
        return self.body(self.inp(x))

    def keep_inputs(self, keep):
        # Bo cac gene bi loai khoi lop dau vao, giu nguyen trong so cua gene con lai
        keep_t = torch.as_tensor(keep, device=self.inp.weight.device)
        new = nn.Linear(len(keep), self.inp.out_features).to(self.inp.weight.device)
        with torch.no_grad():
            new.weight.copy_(self.inp.weight[:, keep_t])
            new.bias.copy_(self.inp.bias)
        self.inp = new


def _train(net, Xt, yt, cw, epochs):
    opt = torch.optim.Adam(net.parameters(), lr=SCFSNN_LR)
    net.train()
    n = len(yt)
    for _ in range(epochs):
        perm = torch.randperm(n, device=DEVICE)
        for s in range(0, n, SCFSNN_BATCH):
            b = perm[s:s + SCFSNN_BATCH]
            if len(b) < 2:          # BatchNorm can >= 2 mau
                continue
            xb = Xt[b]
            if SCFSNN_INPUT_NOISE > 0:     # eta trong code goc: batch_xs = X + randn * eta
                xb = xb + SCFSNN_INPUT_NOISE * torch.randn_like(xb)
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(net(xb), yt[b], weight=cw).backward()
            opt.step()


def _importance(net, Xt, yt, cw):
    # S_j = trung binh |dL(y_i, O_i) / dx_ij| tren tap holdout (cong thuc (2) trong bai bao)
    net.eval()
    Xg = Xt.clone().requires_grad_(True)
    loss = F.cross_entropy(net(Xg), yt, weight=cw, reduction="sum")
    grad, = torch.autograd.grad(loss, Xg)
    return grad.abs().mean(0).cpu().numpy().astype(float)


def select_features(X, y, k):
    _gpu_begin()
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    classes, yc = np.unique(y, return_inverse=True)
    Xp = _preprocess(X)
    n, p = Xp.shape
    q = int(p * SCFSNN_Q_RATIO)

    # 1. q surrogate gene (biet chac la nhieu): lay mau khong hoan lai tu toan bo ma tran du lieu
    pool = Xp.ravel()
    Xa = np.hstack([Xp, pool[rng.choice(pool.size, size=n * q, replace=False)].reshape(n, q)])

    # 2. Tach train cua fold -> fit / holdout (holdout dung tinh importance va FDR, giong tap "validation" cua bai bao)
    fit_i, hold_i = train_test_split(np.arange(n), test_size=SCFSNN_HOLDOUT, stratify=yc, random_state=SEED)
    Xf, yf = Xa[fit_i], yc[fit_i]

    # 3. Tang cuong du lieu convex pseudo-data (CPD): x = (1 - a) x_i + a x_j, a ~ U(0, d), nhan y_i
    if SCFSNN_CPD_D > 0:
        j = rng.permutation(len(Xf))
        a = rng.uniform(0, SCFSNN_CPD_D, size=(len(Xf), 1))
        Xf, yf = np.vstack([Xf, (1 - a) * Xf + a * Xf[j]]), np.concatenate([yf, yf])

    Xf_t = torch.tensor(Xf, dtype=torch.float32, device=DEVICE)
    yf_t = torch.tensor(yf, dtype=torch.long, device=DEVICE)
    Xh_t = torch.tensor(Xa[hold_i], dtype=torch.float32, device=DEVICE)
    yh_t = torch.tensor(yc[hold_i], dtype=torch.long, device=DEVICE)
    cw = _class_weight(yc, len(classes))
    del Xa, Xf

    # 4. Khoi tao: train voi tat ca gene, uoc luong p0 = min(2 * #{S_j < median(S_surrogate)}, p)
    active = np.arange(p + q)                 # cot (trong Xa) con trong mang; >= p la surrogate
    net = FSNet(p + q, len(classes)).to(DEVICE)
    _train(net, Xf_t, yf_t, cw, SCFSNN_EPOCHS)
    S = _importance(net, Xh_t, yh_t, cw)
    p0 = max(1, min(2 * int((S[:p] < np.median(S[p:])).sum()), p))

    # 5. Loai nguoc: moi vong xoa ceil(eps * (1 - eta*/eta_hat) * r0) gene co S nho nhat (goc lan surrogate).
    #    Bai bao dung khi eta_hat <= eta*. De co DUNG k gene, notebook: (a) khong bao gio de so gene goc < k,
    #    (b) neu da dat nguong FDR ma van > k gene goc thi tiep tuc loai gene goc co S nho nhat (ceil(eps * (so du))).
    n_orig_at_fdr, it = None, 0
    while it < SCFSNN_MAX_ITER:
        sur = active >= p
        r0, n_orig = int(sur.sum()), int((~sur).sum())
        fdr = (r0 / q) * p0 / max(n_orig, 1)
        if fdr <= SCFSNN_FDR and n_orig_at_fdr is None:
            n_orig_at_fdr = n_orig
        if n_orig <= k:
            break
        if fdr > SCFSNN_FDR:
            m = math.ceil(SCFSNN_ELIM_RATE * (1 - SCFSNN_FDR / fdr) * r0)
            cand = np.argsort(S, kind="stable")
        else:
            m = math.ceil(SCFSNN_ELIM_RATE * (n_orig - k))
            cand = np.flatnonzero(~sur)[np.argsort(S[~sur], kind="stable")]
        drop, orig_left = [], n_orig - k
        for j in cand:
            if len(drop) == m:
                break
            if not sur[j]:
                if orig_left == 0:
                    continue
                orig_left -= 1
            drop.append(j)
        if not drop:
            break
        keep = np.setdiff1d(np.arange(len(active)), drop)
        keep_t = torch.as_tensor(keep, device=DEVICE)
        active = active[keep]
        net.keep_inputs(keep)
        Xf_t, Xh_t = Xf_t[:, keep_t], Xh_t[:, keep_t]
        it += 1
        _train(net, Xf_t, yf_t, cw, SCFSNN_EPOCHS)
        S = _importance(net, Xh_t, yh_t, cw)
        if it % 10 == 0:
            print(f"    vong {it:3d} | gene goc {int((active < p).sum()):6d} | surrogate {int((active >= p).sum()):5d} "
                  f"| FDR uoc luong {fdr:.3f}")

    # 6. Bo surrogate con lai, train them tren k gene goc roi xep hang theo S
    orig_pos = np.flatnonzero(active < p)
    pos_t = torch.as_tensor(orig_pos, device=DEVICE)
    net.keep_inputs(orig_pos)
    Xf_t, Xh_t = Xf_t[:, pos_t], Xh_t[:, pos_t]
    _train(net, Xf_t, yf_t, cw, SCFSNN_FINAL_EPOCHS)
    S = _importance(net, Xh_t, yh_t, cw)
    _gpu_end()

    order = np.argsort(-S, kind="stable")[:k]
    SELECT_INFO.update(p0_hat=p0, n_surrogate=q, n_iterations=it, fdr_at_k=float(fdr),
                       n_orig_at_fdr_cutoff=n_orig_at_fdr)
    return active[orig_pos][order], S[order]

## Chạy trên 5 fold

In [ ]:
all_rows, metric_rows = [], []

for fold in range(1, N_FOLDS + 1):
    X, y, genes, _ = load_split(fold, "train")   # CHI dung tap train cho feature selection
    k = min(N_FEATURES, X.shape[1])

    with ResourceTracker(f"{METHOD}_{DATASET_NAME}_fold{fold}") as rt:
        idx, scores = select_features(X, y, k)

    sel = pd.DataFrame({"fold": fold, "rank": np.arange(1, len(idx) + 1),
                        "gene": genes[idx], "score": scores})
    sel.to_csv(os.path.join(RESULT_DIR, "selected_genes", f"fold_{fold}.csv"), index=False)
    all_rows.append(sel)

    m = {"dataset": DATASET_NAME, "method": METHOD, "fold": fold,
         "n_train": X.shape[0], "n_genes_total": X.shape[1], "n_selected": len(idx),
         **SELECT_INFO, **rt.metrics}
    metric_rows.append(m)
    print(f"fold {fold}: {X.shape} -> {len(idx)} gene | {m['time_sec']:.2f}s | CPU {m['cpu_util_percent']:.1f}% "
          f"({m['avg_cores_used']:.2f} core) | peak {m['peak_memory_mb']:.0f} MB | "
          f"{m['energy_kwh']:.3e} kWh | {m['carbon_g_co2eq']:.4f} gCO2eq | top5: {list(sel.gene[:5])}")
    del X, y

selected = pd.concat(all_rows, ignore_index=True)
selected.to_csv(os.path.join(RESULT_DIR, "selected_genes_all_folds.csv"), index=False)
with open(os.path.join(RESULT_DIR, "selected_genes.json"), "w") as f:
    json.dump({f"fold_{r}": g.gene.tolist() for r, g in selected.groupby("fold")}, f, indent=1)

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(os.path.join(RESULT_DIR, "metrics_per_fold.csv"), index=False)
metrics

## Tổng hợp độ đo

In [ ]:
cols = ["time_sec", "cpu_time_sec", "cpu_util_percent", "avg_cores_used", "system_cpu_percent_mean",
        "peak_memory_mb", "peak_memory_increase_mb", "energy_kwh", "energy_joule",
        "edp_joule_sec", "carbon_kg_co2eq", "carbon_g_co2eq"]
summary = metrics[cols].agg(["mean", "std", "min", "max"]).T
summary.loc["time_sec", "total"] = metrics["time_sec"].sum()
summary.loc["energy_kwh", "total"] = metrics["energy_kwh"].sum()
summary.loc["carbon_kg_co2eq", "total"] = metrics["carbon_kg_co2eq"].sum()
summary.to_csv(os.path.join(RESULT_DIR, "metrics_summary.csv"))

with open(os.path.join(RESULT_DIR, "config.json"), "w") as f:
    json.dump({"dataset": DATASET_NAME, "data_dir": DATA_DIR, "method": METHOD, "n_features": N_FEATURES,
               "n_folds": N_FOLDS, "seed": SEED, "params": METHOD_PARAMS, "cpu_count": os.cpu_count(),
               "country_iso_code": COUNTRY_ISO_CODE}, f, indent=1)
summary

## Độ ổn định tập gene giữa các fold (tham khảo)

In [ ]:
# Tham khao nhanh: do on dinh tap gene giua cac fold (Jaccard cap doi, trung binh)
sets = {r: set(g.gene) for r, g in selected.groupby("fold")}
folds = sorted(sets)
jac = pd.DataFrame(index=folds, columns=folds, dtype=float)
for a in folds:
    for b in folds:
        jac.loc[a, b] = len(sets[a] & sets[b]) / len(sets[a] | sets[b])
pairs = [jac.loc[a, b] for i, a in enumerate(folds) for b in folds[i + 1:]]
print(f"Jaccard trung binh giua cac fold: {np.mean(pairs):.3f}")
freq = selected.gene.value_counts()
print(f"Gene xuat hien o ca {N_FOLDS} fold: {int((freq == N_FOLDS).sum())}")
freq.rename("n_folds").to_csv(os.path.join(RESULT_DIR, "gene_frequency.csv"))
jac.round(3)

In [ ]:
import shutil
zip_path = shutil.make_archive(f"/kaggle/working/fs_{METHOD}_{DATASET_NAME}", "zip", RESULT_DIR)
print("Da nen:", zip_path)
for root, _, files in os.walk(RESULT_DIR):
    for fn in sorted(files):
        print(" ", os.path.relpath(os.path.join(root, fn), RESULT_DIR))